In [ ]:
import json
import os
import re
import shutil
import warnings
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")


In [ ]:

# =========================
# SETTINGS
# =========================
CELL_SIZE = 500
RANDOM_STATE = 42
USE_SUPERVISED = True

# Включать сравнение с baseline только если в таблице действительно есть baseline_score.
COMPARE_WITH_BASELINE = True
MIN_BASELINE_CORR = 0.65

# ML mode:
# "evidence_points" — обучаемся по известным точкам/аномалиям.
ML_MODE = "evidence_points"

# Положительные ячейки — не только сама точка, а небольшая окрестность точки/аномалии.
# Увеличено относительно прежнего варианта: это делает синие зоны более геологически связными.
POSITIVE_BUFFER_CELLS = 2.6

# Отрицательный класс берём достаточно далеко от известных точек,
# чтобы не записать потенциально перспективные соседние ячейки в ложные нули.
NEGATIVE_DISTANCE_CELLS = 8.5

# Пространственные блоки для честной проверки.
SPATIAL_BLOCK_CELLS = 12
MIN_POSITIVE_CELLS = 8

# Модели. Используется blend RandomForest + ExtraTrees:
# RF даёт устойчивость, ExtraTrees делает поверхность резче и лучше выделяет компактные зоны.
RF_N_ESTIMATORS = 350
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 4
RF_MIN_SAMPLES_SPLIT = 8

ET_N_ESTIMATORS = 350
ET_MAX_DEPTH = 11
ET_MIN_SAMPLES_LEAF = 3
ET_MIN_SAMPLES_SPLIT = 6

# Для cross-validation используем облегчённые модели, чтобы ноутбук не висел.
CV_N_ESTIMATORS = 160

# Итоговая смесь.
# Важно: ML не должен полностью "перекричать" геологическую факторную модель,
# иначе карта получается пятнистой. Поэтому финальная поверхность — ML + геологический якорь.
W_ML = 0.42
W_GEO = 0.43
W_COINCIDENCE = 0.11
W_LOCAL = 0.04

# Резкость итоговой поверхности.
# >1 подавляет слабые зоны и делает сильные синие зоны более компактными.
PROSPECTIVITY_POWER = 1.45

# Сглаживание: 0 = совсем без сглаживания, 1-2 = мягкое сглаживание.
# Для карты как в ответе нужно не 0, а умеренное сглаживание после формирования поверхности.
GEO_SMOOTH_PASSES = 2
ML_SMOOTH_PASSES = 1
FINAL_SMOOTH_PASSES = 2

# Квантили расстояний для proximity.
# Эти значения сильнее выделяют зоны около палеодолин/разломов/структур и слабее раздувают дайки.
Q_FACIES = 0.74
Q_PALEO = 0.70
Q_STRUCT = 0.68
Q_MAGM = 0.36
Q_TECT1 = 0.69
Q_TECT2 = 0.69

# Штрафы: нужны, чтобы убрать синие ложные пятна на краях и вдоль одиноких разломов.
TECT_ONLY_PENALTY_WEIGHT = 0.10
EDGE_PENALTY_WEIGHT = 0.09
BORDER_DISTANCE_CELLS = 2.8

N_DISPLAY_CLASSES = 20
SHOW_POINTS = True


In [ ]:

# =========================
# PATHS
# =========================
def maybe_unpack_zip():
    """Если ноутбук запущен в /mnt/data и рядом есть Прогноз.zip, распаковываем его."""
    zip_candidates = [
        Path("/mnt/data/Прогноз.zip"),
        Path.cwd() / "Прогноз.zip",
    ]
    out_dir = Path("/mnt/data/prog_zip")
    for zp in zip_candidates:
        if zp.exists():
            shp_dir = out_dir / "shp_dbf"
            if not (shp_dir.exists() and (shp_dir / "svita_new.shp").exists()):
                out_dir.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(zp, "r") as z:
                    z.extractall(out_dir)
            return out_dir
    return None

def find_base_dir() -> Path:
    unpacked = maybe_unpack_zip()
    candidates = [
        Path.cwd(),
        unpacked if unpacked is not None else Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        if base is None:
            continue
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Положи рядом папку shp_dbf или архив Прогноз.zip.")

BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "ml_forecast_result_corrected"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "forecast_ml_corrected.gpkg"
OUT_PNG = OUT_DIR / "forecast_ml_corrected.png"
OUT_COMPARE = OUT_DIR / "compare_ml_corrected.png"
OUT_PROX = OUT_DIR / "prox_magm_ml_corrected.png"
OUT_CSV = OUT_DIR / "grid_ml_corrected.csv"
OUT_JSON = OUT_DIR / "metrics_ml_corrected.json"

print("BASE_DIR:", BASE_DIR)
print("SHP_DIR:", SHP_DIR)
print("OUT_DIR:", OUT_DIR)


In [5]:

# =========================
# HELPERS
# =========================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def contrast_power(values, power=1.0):
    """Повышает контраст: слабые значения давит, сильные оставляет."""
    arr = np.clip(np.asarray(values, dtype=float), 0, 1)
    if power is None or np.isclose(power, 1.0):
        return arr
    return np.power(arr, power)

def sigmoid_contrast(values, center=0.50, steepness=7.0):
    """S-образный контраст без грубого threshold."""
    arr = np.clip(np.asarray(values, dtype=float), 0, 1)
    return 1.0 / (1.0 + np.exp(-steepness * (arr - center)))

def top_overlap(a, b, q=0.90):
    """Доля пересечения верхних q-квантилей двух карт."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() == 0:
        return np.nan
    aa = a[ok]
    bb = b[ok]
    ma = aa >= np.quantile(aa, q)
    mb = bb >= np.quantile(bb, q)
    den = max(int(ma.sum()), 1)
    return float((ma & mb).sum() / den)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    return (np.isfinite(arr) & (filled >= locmax))[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, _ = ndimage.label(arr, structure=structure)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0]) & (labeled > 0)
    return keep[rows, cols]

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias}_shp.pj4")
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases

def load_layer(path: Path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    d = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        d[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = d
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        t = np.sqrt(d)
    elif transform == "cbrt":
        t = np.cbrt(d)
    else:
        t = d
    scale = float(np.nanquantile(t, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(t)), 1.0)
    return np.clip(np.exp(-t / scale), 0, 1)

def collect_points(mask_crs, aliases):
    base_names = {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}
    layers = []
    for name, shp_path in aliases.items():
        if name in base_names:
            continue
        gdf = to_crs_safe(load_layer(shp_path), mask_crs)
        types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in types or "MultiPoint" in types:
            layers.append(gdf)
    if not layers:
        return None
    pts = pd.concat(layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def make_display_classes(grid):
    # Для cmap="bwr_r": высокий display_score отображается синим.
    disp = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(disp, bins[1:-1], right=False)
    return grid

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prospectivity"].quantile(0.97))
    q_support = float(grid["prospectivity"].quantile(0.88))
    q_local = float(grid["local_bonus"].quantile(0.97))
    q_coinc = float(grid["coincidence_score"].quantile(0.94))
    q_magm = float(grid["prox_magm"].quantile(0.90))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.84))

    local_peak = local_max_mask(grid, "prospectivity", shape)
    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    support_zone = grid["prospectivity"] >= q_support

    core_gold = (
        support_zone &
        local_peak &
        (grid["prospectivity"] >= q_best) &
        (
            ((grid["local_bonus"] >= q_local) & (grid["tect_magm_intersection"] >= q_tmagm)) |
            ((grid["coincidence_score"] >= q_coinc) & (grid["prox_magm"] >= q_magm))
        )
    )

    linear_gold = (
        support_zone &
        (grid["prospectivity"] >= float(grid["prospectivity"].quantile(0.94))) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["coincidence_score"] >= q_coinc)
    )

    edge_linear_gold = linear_gold & (grid["dist_to_boundary"] <= CELL_SIZE * 0.75)

    grid["gold_seed"] = (core_gold | edge_linear_gold).astype(int)
    grid["gold_zone"] = keep_large_components(grid, "gold_seed", shape, min_cells=4).astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    ax.legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(ax, mask)
    ax.set_title("ML-прогноз перспективности")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    axes[1].legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("ML-прогноз перспективности")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


In [6]:
# =========================
# LOAD DATA
# =========================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

In [7]:

# =========================
# GRID + FEATURES
# =========================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

for src, name in [
    (facies, "dist_facies"),
    (paleo, "dist_paleo"),
    (struct, "dist_struct"),
    (magm, "dist_magm"),
    (tect1, "dist_tect1"),
    (tect2, "dist_tect2"),
]:
    grid = add_distance_feature(grid, src, name)

# Proximity: 1 — близко к фактору, 0 — далеко.
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], "cbrt", Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], "cbrt", Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], "sqrt", Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], "sqrt", Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], "cbrt", Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], "cbrt", Q_TECT2)

# Комбинации факторов. Они важнее простого взвешенного среднего:
# рудный узел должен появляться там, где несколько факторов совпадают пространственно.
grid["tect_combo"] = np.sqrt(np.clip(grid["prox_tect1"] * grid["prox_tect2"], 0, 1))
grid["tect_sum"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(np.clip(grid["tect_sum"] * grid["prox_magm"], 0, 1))
grid["tect_struct_intersection"] = np.sqrt(np.clip(grid["tect_sum"] * grid["prox_struct"], 0, 1))
grid["paleo_struct_intersection"] = np.sqrt(np.clip(grid["prox_paleo"] * grid["prox_struct"], 0, 1))
grid["paleo_facies_intersection"] = np.sqrt(np.clip(grid["prox_paleo"] * grid["prox_facies"], 0, 1))

# Ядро совпадения: близость к палеогеоморфологии + структурам + тектонике/магматизму.
combo_core = (
    np.clip(0.55 * grid["prox_paleo"] + 0.45 * grid["prox_facies"], 0, 1)
    * np.clip(0.55 * grid["prox_struct"] + 0.45 * grid["prox_magm"], 0, 1)
    * np.clip(0.65 * grid["tect_sum"] + 0.35 * grid["tect_intersection"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

# "Одинокая тектоника" без поддержки литологии/магматизма/палеодолин часто даёт ложные синие полосы.
tect_support = (
    0.30 * grid["prox_magm"] +
    0.32 * grid["prox_struct"] +
    0.28 * grid["prox_paleo"] +
    0.10 * grid["prox_facies"]
)
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_sum"] - tect_support, 0, 1), 0.02, 0.98)

# Дополнительный компактный признак: минимум "штрафа расстояния" по ключевым факторам.
# Он помогает получить синие зоны как в эталоне, но остаётся признаком для ML, а не ручной картой.
grid["multi_factor_core"] = robust_normalize_01(
    0.24 * grid["paleo_struct_intersection"] +
    0.22 * grid["tect_magm_intersection"] +
    0.20 * grid["tect_struct_intersection"] +
    0.18 * grid["coincidence_score"] +
    0.16 * grid["paleo_facies_intersection"],
    0.02, 0.98,
)


In [8]:

# =========================
# GEOLOGICAL ANCHOR SCORE
# =========================
# Это не финальный ответ, а геологический "якорь", к которому привязывается ML.
# Он нужен, чтобы карта не разваливалась на случайные RF-пятна.
grid["geo_score_raw"] = (
    0.13 * grid["prox_paleo"] +
    0.12 * grid["prox_struct"] +
    0.10 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.11 * grid["prox_tect1"] +
    0.11 * grid["prox_tect2"] +
    0.09 * grid["tect_intersection"] +
    0.09 * grid["tect_magm_intersection"] +
    0.07 * grid["tect_struct_intersection"] +
    0.05 * grid["paleo_struct_intersection"] +
    0.13 * grid["coincidence_score"] +
    0.08 * grid["multi_factor_core"] -
    TECT_ONLY_PENALTY_WEIGHT * grid["tect_only_penalty"]
)

grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=GEO_SMOOTH_PASSES),
    0.02,
    0.98,
)


In [ ]:

# =========================
# ML MODEL: pseudo-labels + spatial validation
# =========================
grid["target"] = np.nan
grid["rf_score"] = grid["geo_score_sm"]
grid["et_score"] = grid["geo_score_sm"]
grid["ml_score"] = grid["geo_score_sm"]
feature_importance = {}
ml_metrics = {}
use_supervised = False

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_sum", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "paleo_facies_intersection",
    "coincidence_score", "multi_factor_core", "tect_only_penalty",
    "geo_score", "geo_score_sm",
    "dist_facies", "dist_paleo", "dist_struct", "dist_magm", "dist_tect1", "dist_tect2",
]

# Координаты добавлены как слабый пространственный контекст.
# Они помогают ML понять региональные зоны, но финальный геологический якорь не даёт им всё испортить.
centroids = grid.geometry.centroid
grid["x_center"] = centroids.x
grid["y_center"] = centroids.y
grid["x_norm"] = robust_normalize_01(grid["x_center"], 0.01, 0.99)
grid["y_norm"] = robust_normalize_01(grid["y_center"], 0.01, 0.99)
feature_cols += ["x_norm", "y_norm"]

# Spatial groups for block cross-validation.
grid["spatial_group"] = (
    (grid["row"] // SPATIAL_BLOCK_CELLS).astype(str) + "_" +
    (grid["col"] // SPATIAL_BLOCK_CELLS).astype(str)
)

def build_pseudo_labels(grid, points):
    """Создаёт обучающие метки: 1 около известных точек/аномалий, 0 далеко от них."""
    target = pd.Series(np.nan, index=grid.index, dtype=float)
    if points is None or len(points) == 0:
        return target, None

    pts_union = unary_union(points.geometry)
    dist_to_points = np.array([geom.distance(pts_union) for geom in grid.geometry], dtype=float)
    pos_dist = CELL_SIZE * POSITIVE_BUFFER_CELLS
    neg_dist = CELL_SIZE * NEGATIVE_DISTANCE_CELLS

    # Положительный класс — окрестность известных проявлений.
    target[dist_to_points <= pos_dist] = 1.0

    # Отрицательный класс — далеко от проявлений и не в верхней части геологического якоря.
    # Это снижает риск назвать потенциальную, но неразведанную зону отрицательной.
    far = dist_to_points >= neg_dist
    low_geo = grid["geo_score_sm"] <= grid["geo_score_sm"].quantile(0.78)
    target[far & low_geo] = 0.0

    return target, dist_to_points

if USE_SUPERVISED:
    grid["target"], dist_to_points = build_pseudo_labels(grid, points)
    if dist_to_points is not None:
        grid["dist_to_evidence"] = dist_to_points

    train_mask = grid["target"].isin([0.0, 1.0])
    pos = int((grid.loc[train_mask, "target"] == 1).sum())
    neg = int((grid.loc[train_mask, "target"] == 0).sum())

    if pos >= MIN_POSITIVE_CELLS and neg > pos:
        X_all = grid[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        X_train_all = X_all.loc[train_mask]
        y_train_all = grid.loc[train_mask, "target"].astype(int)
        groups = grid.loc[train_mask, "spatial_group"]

        auc_values, ap_values = [], []
        unique_groups = groups.nunique()
        n_splits = min(3, unique_groups)

        if n_splits >= 2 and y_train_all.nunique() == 2:
            cv = GroupKFold(n_splits=n_splits)
            for tr_idx, te_idx in cv.split(X_train_all, y_train_all, groups):
                if y_train_all.iloc[te_idx].nunique() < 2:
                    continue

                rf_cv = RandomForestClassifier(
                    n_estimators=CV_N_ESTIMATORS,
                    max_depth=max(6, RF_MAX_DEPTH - 3),
                    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
                    min_samples_split=RF_MIN_SAMPLES_SPLIT,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )
                et_cv = ExtraTreesClassifier(
                    n_estimators=CV_N_ESTIMATORS,
                    max_depth=max(6, ET_MAX_DEPTH - 3),
                    min_samples_leaf=ET_MIN_SAMPLES_LEAF,
                    min_samples_split=ET_MIN_SAMPLES_SPLIT,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )

                rf_cv.fit(X_train_all.iloc[tr_idx], y_train_all.iloc[tr_idx])
                et_cv.fit(X_train_all.iloc[tr_idx], y_train_all.iloc[tr_idx])
                prob = 0.55 * rf_cv.predict_proba(X_train_all.iloc[te_idx])[:, 1] + 0.45 * et_cv.predict_proba(X_train_all.iloc[te_idx])[:, 1]

                auc_values.append(roc_auc_score(y_train_all.iloc[te_idx], prob))
                ap_values.append(average_precision_score(y_train_all.iloc[te_idx], prob))

        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        et = ExtraTreesClassifier(
            n_estimators=ET_N_ESTIMATORS,
            max_depth=ET_MAX_DEPTH,
            min_samples_leaf=ET_MIN_SAMPLES_LEAF,
            min_samples_split=ET_MIN_SAMPLES_SPLIT,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        rf.fit(X_train_all, y_train_all)
        et.fit(X_train_all, y_train_all)

        rf_prob = rf.predict_proba(X_all)[:, 1]
        et_prob = et.predict_proba(X_all)[:, 1]

        grid["rf_score"] = robust_normalize_01(rf_prob, 0.02, 0.98)
        grid["et_score"] = robust_normalize_01(et_prob, 0.02, 0.98)
        grid["ml_score"] = robust_normalize_01(0.55 * grid["rf_score"] + 0.45 * grid["et_score"], 0.02, 0.98)

        rf_imp = pd.Series(rf.feature_importances_, index=feature_cols)
        et_imp = pd.Series(et.feature_importances_, index=feature_cols)
        imp = (0.55 * rf_imp + 0.45 * et_imp).sort_values(ascending=False)
        feature_importance = imp.to_dict()

        use_supervised = True

        ml_metrics = {
            "ml_mode": ML_MODE,
            "positive_train_cells": pos,
            "negative_train_cells": neg,
            "spatial_cv_auc_mean": None if len(auc_values) == 0 else float(np.mean(auc_values)),
            "spatial_cv_auc_std": None if len(auc_values) == 0 else float(np.std(auc_values)),
            "spatial_cv_ap_mean": None if len(ap_values) == 0 else float(np.mean(ap_values)),
            "spatial_cv_ap_std": None if len(ap_values) == 0 else float(np.std(ap_values)),
            "spatial_cv_folds_used": int(len(auc_values)),
        }
    else:
        ml_metrics = {
            "ml_mode": ML_MODE,
            "positive_train_cells": pos,
            "negative_train_cells": neg,
            "warning": "Недостаточно обучающих ячеек; используется geo_score_sm как fallback.",
        }

grid["rf_score_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "rf_score", grid_shape, passes=ML_SMOOTH_PASSES),
    0.02,
    0.98,
)
grid["et_score_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "et_score", grid_shape, passes=ML_SMOOTH_PASSES),
    0.02,
    0.98,
)
grid["ml_score_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "ml_score", grid_shape, passes=ML_SMOOTH_PASSES),
    0.02,
    0.98,
)


In [ ]:

# =========================
# FINAL ML SURFACE
# =========================
# Локальный бонус — не просто "близость к одному объекту", а совпадение нескольких факторов.
grid["local_bonus"] = robust_normalize_01(
    0.24 * grid["tect_intersection"] +
    0.25 * grid["tect_magm_intersection"] +
    0.23 * grid["tect_struct_intersection"] +
    0.18 * grid["paleo_struct_intersection"] +
    0.10 * grid["multi_factor_core"],
    0.02,
    0.98,
)

# Штраф края: на первой версии карты синие пятна часто прилипали к границе маски.
mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * BORDER_DISTANCE_CELLS))
grid["edge_false_penalty"] = robust_normalize_01(edge_near * np.clip(grid["tect_only_penalty"], 0, 1), 0.02, 0.98)

if use_supervised:
    # Главный исправленный вариант:
    # 1) ML даёт адаптацию к evidence points.
    # 2) geo_score_sm удерживает форму как в методике.
    # 3) coincidence/local делают синие области компактными.
    grid["prospectivity_raw"] = (
        W_ML * grid["ml_score_sm"] +
        W_GEO * grid["geo_score_sm"] +
        W_COINCIDENCE * grid["coincidence_score"] +
        W_LOCAL * grid["local_bonus"]
    )
else:
    grid["prospectivity_raw"] = (
        0.62 * grid["geo_score_sm"] +
        0.23 * grid["coincidence_score"] +
        0.15 * grid["local_bonus"]
    )

grid["prospectivity_raw"] = (
    grid["prospectivity_raw"]
    - TECT_ONLY_PENALTY_WEIGHT * grid["tect_only_penalty"]
    - EDGE_PENALTY_WEIGHT * grid["edge_false_penalty"]
)

# Умеренное финальное сглаживание: убирает пиксельную пятнистость, но не размывает всё как раньше.
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(
    grid,
    "prospectivity_raw",
    grid_shape,
    passes=FINAL_SMOOTH_PASSES,
)

# Контраст: слабые/средние значения уходят в красное/белое, сильные ядра становятся синими.
p0 = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)
p1 = contrast_power(p0, PROSPECTIVITY_POWER)

# Мягкая S-кривая делает зоны ближе к эталонной картинке: меньше мелких синих пятен, больше цельных областей.
p2 = sigmoid_contrast(p1, center=0.43, steepness=6.5)

grid["prospectivity"] = robust_normalize_01(p2, 0.02, 0.98)

# Для совместимости с методичкой: чем меньше prognoz, тем выше перспективность.
grid["prognoz"] = 1.0 - grid["prospectivity"]

# Более понятное имя для ML-проекта.
grid["ml_prospectivity"] = grid["prospectivity"]

# Baseline diagnostics, если baseline_score уже был создан в другом варианте ноутбука.
baseline_corr = np.nan
baseline_top5_overlap = np.nan
baseline_top10_overlap = np.nan
if "baseline_score" in grid.columns:
    baseline_arr = grid["baseline_score"].to_numpy()
    new_arr = grid["prospectivity"].to_numpy()
    ok = np.isfinite(baseline_arr) & np.isfinite(new_arr)
    if ok.sum() > 2:
        baseline_corr = float(np.corrcoef(baseline_arr[ok], new_arr[ok])[0, 1])
        baseline_top5_overlap = top_overlap(baseline_arr, new_arr, 0.95)
        baseline_top10_overlap = top_overlap(baseline_arr, new_arr, 0.90)

if COMPARE_WITH_BASELINE and np.isfinite(baseline_corr) and baseline_corr < MIN_BASELINE_CORR:
    print(f"ВНИМАНИЕ: корреляция с baseline = {baseline_corr:.3f}, ниже порога {MIN_BASELINE_CORR:.2f}")

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

print(f"ML mode: {ML_MODE}")
if np.isfinite(baseline_corr):
    print(f"Baseline correlation: {baseline_corr:.3f}")
    print(f"Top-5% overlap: {baseline_top5_overlap:.3f}; Top-10% overlap: {baseline_top10_overlap:.3f}")


In [ ]:

# =========================
# SAVE
# =========================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "method": "ML blend: RandomForest + ExtraTrees + geological anchor + compactness correction",
    "ml_mode": ML_MODE,
    "positive_cells": int((grid["target"] == 1).sum()),
    "negative_cells": int((grid["target"] == 0).sum()),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "baseline_correlation": None if not np.isfinite(baseline_corr) else float(baseline_corr),
    "baseline_top5_overlap": None if not np.isfinite(baseline_top5_overlap) else float(baseline_top5_overlap),
    "baseline_top10_overlap": None if not np.isfinite(baseline_top10_overlap) else float(baseline_top10_overlap),
    "ml_metrics": ml_metrics,
    "point_count": int(len(points)) if points is not None else 0,
    "feature_importance": feature_importance,
    "settings": {
        "W_ML": W_ML,
        "W_GEO": W_GEO,
        "W_COINCIDENCE": W_COINCIDENCE,
        "W_LOCAL": W_LOCAL,
        "PROSPECTIVITY_POWER": PROSPECTIVITY_POWER,
        "GEO_SMOOTH_PASSES": GEO_SMOOTH_PASSES,
        "ML_SMOOTH_PASSES": ML_SMOOTH_PASSES,
        "FINAL_SMOOTH_PASSES": FINAL_SMOOTH_PASSES,
    },
}
OUT_JSON.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print(grid[["ml_prospectivity", "prognoz", "rf_score_sm", "et_score_sm", "gold_zone"]].describe())
print("Supervised ML:", use_supervised)
print("ML metrics:", ml_metrics)

if feature_importance:
    print("Top feature importance:")
    for k, v in list(feature_importance.items())[:12]:
        print(f"  {k}: {v:.4f}")
